# CSNAnalysis Tutorial 2: Committors and Visualization

**Dickson Lab, Michigan State University**

---

## What is a committor probability?

Given a set of metastable states ("basins"), the **committor probability** $p_A(i)$ for
state $i$ is the probability that a trajectory starting in $i$ will reach basin $A$
before any other basin.  Committor probabilities partition every state according to
where it is "headed" and sum to 1 across all basins:

$$\sum_\alpha p_\alpha(i) = 1 \quad \forall\, i$$

States deep inside basin $A$ have $p_A \approx 1$; transition-state
regions have $p_A \approx p_B \approx 0.5$.

**This notebook covers:**
1. 2-basin committors via linear algebra
2. N-basin committors via iterative multiplication
3. Interpreting and plotting committors
4. Extracting slow eigenvectors
5. Coloring nodes by committor and exporting to Gephi

## Setup

We reproduce the trimmed CSN from Notebook 1 in a few lines.

In [ ]:
import numpy as np
import scipy.sparse
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline

from csnanalysis.csn import CSN

count_mat = scipy.sparse.load_npz('matrix.npz')
our_csn = CSN(count_mat, symmetrize=True)
our_csn.trim(by_inflow=True, by_outflow=True, min_count=20)
_ = our_csn.calc_eig_weights()   # stored as node attribute 'eig_weights'

print(f"Trimmed CSN: {our_csn.trim_nnodes} nodes")

## 1. Defining basins

Basins are user-supplied lists of state indices.  They are typically identified
from clustering analysis or domain knowledge about the system.  Here we use three
basins — $A$, $B$, and $U$ — representing two structural endpoints and an
unfolded/intermediate ensemble.

In [ ]:
Astates = [2031, 596, 1923, 3223, 2715]
Bstates = [1550, 3168, 476, 1616, 2590]
Ustates = list(np.loadtxt('state_U.dat', dtype=int))

print(f"Basin A: {len(Astates)} states")
print(f"Basin B: {len(Bstates)} states")
print(f"Basin U: {len(Ustates)} states")

## 2. Two-basin committors (linear algebra method)

For exactly two basins, the committor equation

$$q_i = \sum_j T_{ij}\, q_j \qquad i \notin A \cup B$$

is a linear system $A\mathbf{q} = \mathbf{b}$ that can be solved directly.
This gives machine-precision results and is recommended when only two basins are needed.

In [ ]:
comms_2 = our_csn.calc_committors(
    [Astates, Bstates],
    labels=['pA_2basin', 'pB_2basin'],
    method='linalg'
)

print(f"Committor array shape: {comms_2.shape}  (n_states × n_basins)")
print(f"Sum check (should all be 1): min = {comms_2.sum(axis=1).min():.6f}, "
      f"max = {comms_2.sum(axis=1).max():.6f}")

In [ ]:
# Interpret a specific state
i = our_csn.trim_indices[100]
print(f"State {i}: pA = {comms_2[i,0]:.3f}, pB = {comms_2[i,1]:.3f}")
print(f"  → a trajectory starting here reaches A first {comms_2[i,0]:.1%} of the time")

## 3. Three-basin committors (iterative method)

For more than two basins, `CSNAnalysis` uses an iterative approach: a modified
transition matrix $S$ is constructed where each basin state acts as an absorbing sink.
Repeated squaring of $S$ converges to $S^\infty$, whose columns give the absorption
probabilities from every starting state.

In [ ]:
basins = [Astates, Bstates, Ustates]
labels = ['pA', 'pB', 'pU']

comms = our_csn.calc_committors(basins, labels=labels, add_basins=True,
                                 basin_labels=['basin_A', 'basin_B', 'basin_U'])

print(f"Committor shape: {comms.shape}")
i = our_csn.trim_indices[0]
print(f"\nState {i}: pA={comms[i,0]:.3f}, pB={comms[i,1]:.3f}, pU={comms[i,2]:.3f}")
print(f"  → most likely to commit to "
      f"{'A' if comms[i,0]>comms[i,1] and comms[i,0]>comms[i,2] else 'B' if comms[i,1]>comms[i,2] else 'U'}")

## 4. Plotting committors

Plotting pairs of committors reveals the structure of the free energy landscape.
States with $p_A + p_B \approx 1$ and $p_U \approx 0$ lie on direct $A \leftrightarrow B$
pathways; states with high $p_U$ are part of the unfolded ensemble.

In [ ]:
wts = our_csn.calc_eig_weights()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

pairs = [('pA', 'pB', 0, 1), ('pA', 'pU', 0, 2), ('pB', 'pU', 1, 2)]
for ax, (xl, yl, xi, yi) in zip(axes, pairs):
    # weight point sizes by equilibrium population
    sz = 1 + 200 * wts / wts.max()
    ax.scatter(comms[:, xi], comms[:, yi], s=sz, alpha=0.3, color='steelblue')
    ax.set_xlabel(xl, fontsize=12)
    ax.set_ylabel(yl, fontsize=12)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle("Committor probabilities (point size ∝ equilibrium weight)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Slow eigenvectors

The eigenvectors of $T$ associated with eigenvalues close to 1 describe the
**slowest relaxation modes** of the system.  The corresponding eigenvalues are
related to implied timescales: $\tau_k = -\Delta t / \ln |\lambda_k|$.

The leading eigenvector (eigenvalue = 1) is the stationary distribution.  The
second and third eigenvectors identify which states change population most slowly.

In [ ]:
eig_vecs, eig_vals, _, _ = our_csn.calc_eigvectors(n_eig=3, save_label='eig')

print("Top 3 sub-leading eigenvalues (excluding stationary mode):")
for k, lam in enumerate(eig_vals):
    print(f"  λ{k+1} = {lam:.6f}")

In [ ]:
# Compare the first eigenvector against pA − pB
# (the slowest mode often aligns with the dominant committor difference)
in_trim = np.array(our_csn.trim_indices)
delta_comm = comms[in_trim, 0] - comms[in_trim, 1]   # pA - pB for trimmed states

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(delta_comm, eig_vecs[in_trim, 0], alpha=0.3, s=4, color='coral')
ax.set_xlabel("pA − pB", fontsize=12)
ax.set_ylabel("1st eigenvector", fontsize=12)
ax.set_title("Slow mode vs. committor difference")
plt.tight_layout()
plt.show()

## 6. Coloring nodes by committor

`colors_from_committors` maps committor values to RGB colors.  For three basins
the three channels map naturally to (R, G, B), making the dominant basin assignment
immediately visible in a graph layout.

In [ ]:
rgb = our_csn.colors_from_committors(comms)
our_csn.set_colors(rgb)

# Verify: node 0 should now carry viz/color data
node0 = our_csn.graph.nodes[our_csn.trim_indices[0]]
print(f"Node {our_csn.trim_indices[0]} color: {node0['viz']['color']}")

## 7. Exporting to Gephi

The `.gexf` format encodes all node attributes — weights, committors, colors,
eigenvectors — alongside the graph topology.  After opening in
[Gephi](https://gephi.org/), apply the **Force Atlas 2** layout, size nodes
by `eig_weights`, and the committor-derived colors will appear automatically.

In [ ]:
output_file = 'csn_committors.gexf'
nx.readwrite.gexf.write_gexf(our_csn.graph.to_undirected(), output_file)
print(f"Saved {output_file}")
print(f"Node attributes exported: {list(our_csn.graph.nodes[our_csn.trim_indices[0]].keys())}")

The image below shows what a Gephi layout (Force Atlas 2) looks like after
sizing nodes by `eig_weights` and using the committor-derived RGB colors:

![Gephi graph export](committor_net_3state.png)

Red nodes commit preferentially to basin $A$, green to basin $B$, and blue to
basin $U$.  The committor values reveal how the network is partitioned into
kinetically distinct regions.

---
**Next:** [03_mfpt_and_hubscores.ipynb](03_mfpt_and_hubscores.ipynb) —
mean first passage times, first passage time distributions, and hub scores.